# 🧠 RLHF (Reinforcement Learning from Human Feedback) — Complete Demo

**Pure PyTorch implementation — no TRL dependency.**

This notebook demonstrates the full RLHF pipeline:

```
┌────────────┐    ┌──────────────┐    ┌────────────────┐
│  Pretrained │───▶│ Reward Model │───▶│  RLHF-Tuned    │
│   GPT-2     │    │  (Sentiment) │    │     GPT-2      │
└────────────┘    └──────────────┘    └────────────────┘
      │                                       │
      ▼                                       ▼
 "Before RLHF"                         "After RLHF"
 (uncontrolled)                   (steered toward positive)
```

**Algorithm:** REINFORCE with KL penalty (the core mechanism inside PPO)

**Runtime:** GPU recommended (T4 is sufficient). Go to **Runtime → Change runtime type → T4 GPU**

---
## Cell 1 — Install Dependencies

In [1]:
!pip install -q transformers accelerate datasets torch numpy pandas matplotlib seaborn
print("✅ All packages installed!")

✅ All packages installed!


---
## Cell 2 — Imports & Device Setup

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import textwrap
import time
import copy
from collections import defaultdict

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    set_seed,
)

warnings.filterwarnings("ignore")
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device : {device}")


---
## Cell 3 — Load Models

| Model | Role | Details |
|-------|------|---------|
| **GPT-2** (124M) | Policy model | The model we fine-tune with RLHF |
| **GPT-2** (frozen copy) | Reference model | Anchor for KL penalty — prevents catastrophic drift |
| **DistilBERT-SST2** | Reward model | Sentiment classifier used as the reward signal |

In [ ]:
MODEL_NAME  = "gpt2"
REWARD_NAME = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

print("📦 Loading tokenizer …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("📦 Loading policy model (GPT-2) …")
policy_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
policy_model.train()

print("📦 Loading reference model (frozen GPT-2) …")
ref_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False

print("📦 Loading reward model (DistilBERT-SST2) …")
reward_pipe = pipeline(
    "sentiment-analysis",
    model=REWARD_NAME,
    device=device,
    truncation=True,
    max_length=512,
)

total_params = sum(p.numel() for p in policy_model.parameters())
print(f"\n✅ All models loaded!")
print(f"   Policy params : {total_params:,}")

---
## Cell 4 — Prompts & Helper Functions

In [5]:
# ── Evaluation prompts (used BEFORE and AFTER to compare) ──
EVAL_PROMPTS = [
    "The future of artificial intelligence is",
    "Today I feel really",
    "The best thing about working hard is",
    "Climate change will",
    "My experience with this product was",
    "The movie I watched last night was",
    "In the next ten years, technology will",
    "The restaurant we visited was",
    "Life is all about",
    "The new policy announced by the government is",
]

# ── Training prompts (used during RLHF optimization) ──
TRAIN_PROMPTS = [
    "I think this is", "The weather today is", "My day has been",
    "This new feature is", "I believe that", "Working together is",
    "The food here is", "My opinion about this is", "The team did",
    "Looking at the results", "The service was", "I would describe this as",
    "The presentation was", "Overall, I think", "The update made things",
    "I recently tried", "This approach is", "The outcome was",
    "Reflecting on today", "The progress we made is", "I appreciate that",
    "The quality of this is", "My review of this is", "The experience was",
    "What stands out is", "The effort put in was", "The plan seems",
    "I noticed that", "The improvement is", "Going forward, I think",
    "My impression is", "The collaboration was", "This solution is",
    "The feedback was", "I feel confident that", "The design looks",
    "The meeting went", "My take on this is", "The strategy seems",
    "I would say this is",
]


def get_reward(text: str) -> float:
    """Reward = sentiment score. POSITIVE → +score, NEGATIVE → -score."""
    result = reward_pipe(text[:512])[0]
    score = result["score"]
    return score if result["label"] == "POSITIVE" else -score


@torch.no_grad()
def generate_text(model, prompt: str, max_new_tokens: int = 40) -> str:
    """Simple text generation for evaluation."""
    was_training = model.training
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id,
    )
    if was_training:
        model.train()
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


def evaluate_model(model, prompts, label="Model"):
    """Evaluate a model on a list of prompts, return DataFrame with scores."""
    results = []
    for p in prompts:
        gen = generate_text(model, p)
        score = get_reward(gen)
        results.append({"prompt": p, "generated": gen, "reward": score})
    df = pd.DataFrame(results)
    avg = df["reward"].mean()
    print(f"\n{'='*70}")
    print(f"  {label}  —  Average Reward: {avg:.4f}")
    print(f"{'='*70}")
    for _, row in df.iterrows():
        wrapped = textwrap.fill(row["generated"], width=75, subsequent_indent="    ")
        print(f"\n  Prompt : {row['prompt']}")
        print(f"  Output : {wrapped}")
        print(f"  Reward : {row['reward']:.4f}")
    return df

print("✅ Helper functions defined.")
print(f"   Eval prompts  : {len(EVAL_PROMPTS)}")
print(f"   Train prompts : {len(TRAIN_PROMPTS)}")

✅ Helper functions defined.
   Eval prompts  : 10
   Train prompts : 40


---
## Cell 5 — Evaluate PRETRAINED Model (Before RLHF)

This captures the **baseline** — vanilla GPT-2 with no reward-based tuning.

In [ ]:
print("🔍 Evaluating pretrained GPT-2 BEFORE RLHF …\n")
df_before = evaluate_model(policy_model, EVAL_PROMPTS, label="BEFORE RLHF (Pretrained GPT-2)")

---
## Cell 6 — RLHF Training (REINFORCE + KL Penalty)

**Algorithm per training step:**

1. Encode prompt → `input_ids`
2. Generate response tokens autoregressively, collecting log-probs from **policy**
3. Compute log-probs of the same tokens under the frozen **reference** model
4. Score the full text with the **reward model** (sentiment classifier)
5. Per-token KL penalty: `KL = log π_policy - log π_ref`
6. Adjusted reward: `R_adj = R - β × mean(KL)`
7. Policy gradient: `loss = -Σ log π(token) × R_adj`
8. Update policy with Adam + gradient clipping

This is the same core mechanism that PPO uses, simplified for clarity.

In [ ]:
# ─── Hyperparameters ───
NUM_EPOCHS       = 3
BATCH_SIZE       = 8
MAX_NEW_TOKENS   = 30
LEARNING_RATE    = 1.41e-5
KL_COEFF         = 0.15      # β — how strongly to penalize drift from reference
TEMPERATURE      = 0.8
TOP_K            = 50

optimizer = torch.optim.Adam(policy_model.parameters(), lr=LEARNING_RATE)
training_stats = defaultdict(list)
global_step = 0


def generate_with_logprobs(model, input_ids, max_new_tokens, temperature=0.8):
    """
    Generate tokens one-by-one from the policy model,
    collecting per-token log-probabilities for the policy gradient.
    """
    generated_ids = []
    log_probs = []
    current_ids = input_ids.clone()

    for _ in range(max_new_tokens):
        outputs = model(current_ids)
        logits = outputs.logits[:, -1, :] / temperature

        # Top-k filtering
        top_k_logits, top_k_indices = torch.topk(logits, TOP_K, dim=-1)
        filtered_logits = torch.full_like(logits, float("-inf"))
        filtered_logits.scatter_(1, top_k_indices, top_k_logits)

        probs = F.softmax(filtered_logits, dim=-1)
        token_id = torch.multinomial(probs, num_samples=1)

        log_prob = F.log_softmax(filtered_logits, dim=-1)
        token_log_prob = log_prob.gather(1, token_id).squeeze(-1)

        generated_ids.append(token_id.item())
        log_probs.append(token_log_prob.squeeze())

        current_ids = torch.cat([current_ids, token_id], dim=-1)

        if token_id.item() == tokenizer.eos_token_id:
            break

    return generated_ids, log_probs


def compute_ref_logprobs(ref_model, input_ids, generated_ids):
    """Compute log-probs of generated tokens under the frozen reference model."""
    full_ids = torch.cat([
        input_ids,
        torch.tensor([generated_ids], device=device)
    ], dim=-1)

    with torch.no_grad():
        outputs = ref_model(full_ids)
        logits = outputs.logits / TEMPERATURE

    ref_log_probs = []
    prompt_len = input_ids.shape[1]
    for i, token_id in enumerate(generated_ids):
        pos = prompt_len + i - 1
        if i == 0:
            pos = prompt_len - 1
        lp = F.log_softmax(logits[:, pos, :], dim=-1)
        ref_log_probs.append(lp[:, token_id].squeeze())

    return ref_log_probs


# ─── Training Loop ───
print(f"🚀 Starting RLHF Training")
print(f"   Epochs           : {NUM_EPOCHS}")
print(f"   Batch size        : {BATCH_SIZE}")
print(f"   Max new tokens    : {MAX_NEW_TOKENS}")
print(f"   Learning rate     : {LEARNING_RATE}")
print(f"   KL coefficient β  : {KL_COEFF}")
print(f"   Training prompts  : {len(TRAIN_PROMPTS)}")
print("=" * 70)

for epoch in range(NUM_EPOCHS):
    epoch_rewards = []
    epoch_kls = []
    t0 = time.time()

    shuffled_prompts = TRAIN_PROMPTS.copy()
    np.random.shuffle(shuffled_prompts)

    for batch_start in range(0, len(shuffled_prompts), BATCH_SIZE):
        batch_prompts = shuffled_prompts[batch_start : batch_start + BATCH_SIZE]
        batch_rewards = []
        batch_kls = []

        optimizer.zero_grad()

        for prompt in batch_prompts:
            # 1. Tokenize
            input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

            # 2. Generate with log-probs from policy
            policy_model.train()
            gen_ids, policy_logprobs = generate_with_logprobs(
                policy_model, input_ids, MAX_NEW_TOKENS, TEMPERATURE
            )

            if len(gen_ids) == 0:
                continue

            # 3. Reference log-probs
            ref_logprobs = compute_ref_logprobs(ref_model, input_ids, gen_ids)

            # 4. Reward
            full_text = tokenizer.decode(
                list(input_ids.squeeze().cpu().numpy()) + gen_ids,
                skip_special_tokens=True,
            )
            reward = get_reward(full_text)

            # 5. KL divergence (per-token)
            kl_penalties = []
            for p_lp, r_lp in zip(policy_logprobs, ref_logprobs):
                kl_penalties.append(p_lp - r_lp)
            mean_kl = torch.stack(kl_penalties).mean()

            # 6. Adjusted reward
            adjusted_reward = reward - KL_COEFF * mean_kl.item()

            # 7. REINFORCE loss
            policy_loss = torch.tensor(0.0, device=device)
            for lp in policy_logprobs:
                policy_loss = policy_loss - lp * adjusted_reward
            policy_loss = policy_loss / len(policy_logprobs)

            # 8. Backward
            policy_loss.backward()

            batch_rewards.append(reward)
            batch_kls.append(mean_kl.item())

        # Update
        torch.nn.utils.clip_grad_norm_(policy_model.parameters(), max_norm=0.5)
        optimizer.step()

        global_step += 1
        avg_reward = np.mean(batch_rewards) if batch_rewards else 0.0
        avg_kl = np.mean(batch_kls) if batch_kls else 0.0

        epoch_rewards.extend(batch_rewards)
        epoch_kls.extend(batch_kls)

        training_stats["step"].append(global_step)
        training_stats["mean_reward"].append(avg_reward)
        training_stats["mean_kl"].append(avg_kl)
        training_stats["epoch"].append(epoch + 1)

        batch_num = batch_start // BATCH_SIZE + 1
        total_batches = (len(TRAIN_PROMPTS) + BATCH_SIZE - 1) // BATCH_SIZE
        print(
            f"  Epoch {epoch+1}/{NUM_EPOCHS} | "
            f"Batch {batch_num}/{total_batches} | "
            f"Reward: {avg_reward:+.4f} | "
            f"KL: {avg_kl:.4f}"
        )

    elapsed = time.time() - t0
    print(f"\n  ✅ Epoch {epoch+1} — "
          f"Avg Reward: {np.mean(epoch_rewards):+.4f} | "
          f"Avg KL: {np.mean(epoch_kls):.4f} | "
          f"Time: {elapsed:.1f}s\n")

print("=" * 70)
print("🏁 RLHF Training Complete!")

---
## Cell 7 — Evaluate AFTER RLHF

In [ ]:
print("🔍 Evaluating GPT-2 AFTER RLHF …\n")
df_after = evaluate_model(policy_model, EVAL_PROMPTS, label="AFTER RLHF (Tuned GPT-2)")

---
## Cell 8 — Side-by-Side Comparison Table

In [ ]:
print("\n" + "=" * 90)
print("  📊  SIDE-BY-SIDE COMPARISON: BEFORE vs AFTER RLHF")
print("=" * 90)

comparison = pd.DataFrame({
    "Prompt": df_before["prompt"],
    "Reward BEFORE": df_before["reward"].round(4),
    "Reward AFTER": df_after["reward"].round(4),
    "Delta": (df_after["reward"] - df_before["reward"]).round(4),
})

for _, row in comparison.iterrows():
    arrow = "↑" if row["Delta"] > 0 else ("↓" if row["Delta"] < 0 else "→")
    print(f"\n  Prompt: \"{row['Prompt']}\"")
    print(f"    Before: {row['Reward BEFORE']:+.4f}  →  After: {row['Reward AFTER']:+.4f}  ({arrow} {row['Delta']:+.4f})")

improved = (comparison["Delta"] > 0).sum()
worsened = (comparison["Delta"] < 0).sum()

print(f"\n{'─'*90}")
print(f"  Overall Average  BEFORE : {df_before['reward'].mean():+.4f}")
print(f"  Overall Average  AFTER  : {df_after['reward'].mean():+.4f}")
print(f"  Overall Δ               : {df_after['reward'].mean() - df_before['reward'].mean():+.4f}")
print(f"  Prompts improved        : {improved} / {len(comparison)}")
print(f"  Prompts worsened        : {worsened} / {len(comparison)}")
print(f"{'─'*90}\n")

---
## Cell 9 — Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("RLHF Performance Analysis — Before vs After",
             fontsize=16, fontweight="bold", y=1.02)

# ── Plot 1: Per-Prompt Reward Comparison ──
ax1 = axes[0, 0]
x = np.arange(len(EVAL_PROMPTS))
width = 0.35
short_labels = [p[:22] + "…" if len(p) > 22 else p for p in EVAL_PROMPTS]

ax1.bar(x - width/2, df_before["reward"], width, label="Before RLHF", color="#EF6461", alpha=0.85)
ax1.bar(x + width/2, df_after["reward"],  width, label="After RLHF",  color="#60D394", alpha=0.85)
ax1.set_ylabel("Reward Score")
ax1.set_title("Per-Prompt Reward Comparison", fontweight="bold")
ax1.set_xticks(x)
ax1.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=7)
ax1.legend()
ax1.axhline(y=0, color="grey", linewidth=0.5, linestyle="--")
ax1.grid(axis="y", alpha=0.3)

# ── Plot 2: Reward Distribution ──
ax2 = axes[0, 1]
box_data = pd.DataFrame({
    "Reward": list(df_before["reward"]) + list(df_after["reward"]),
    "Stage": ["Before RLHF"] * len(df_before) + ["After RLHF"] * len(df_after),
})
colors_map = {"Before RLHF": "#EF6461", "After RLHF": "#60D394"}
sns.boxplot(data=box_data, x="Stage", y="Reward", palette=colors_map, ax=ax2, width=0.4)
sns.stripplot(data=box_data, x="Stage", y="Reward", color="black", alpha=0.5, ax=ax2, size=6)
ax2.set_title("Reward Distribution", fontweight="bold")
ax2.axhline(y=0, color="grey", linewidth=0.5, linestyle="--")
ax2.grid(axis="y", alpha=0.3)

# ── Plot 3: Training Curves ──
ax3 = axes[1, 0]
stats_df = pd.DataFrame(training_stats)
ax3.plot(stats_df["step"], stats_df["mean_reward"], color="#4D9DE0",
         linewidth=2, marker="o", markersize=4, label="Mean Reward")
ax3.fill_between(stats_df["step"], stats_df["mean_reward"], alpha=0.12, color="#4D9DE0")

ax3_twin = ax3.twinx()
ax3_twin.plot(stats_df["step"], stats_df["mean_kl"], color="#E8A838",
              linewidth=2, marker="s", markersize=3, linestyle="--", label="Mean KL", alpha=0.7)
ax3_twin.set_ylabel("KL Divergence", color="#E8A838")

ax3.set_xlabel("Training Step")
ax3.set_ylabel("Mean Reward", color="#4D9DE0")
ax3.set_title("Training: Reward & KL Over Time", fontweight="bold")
ax3.grid(alpha=0.3)

lines1, labels1 = ax3.get_legend_handles_labels()
lines2, labels2 = ax3_twin.get_legend_handles_labels()
ax3.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=9)

# ── Plot 4: Summary Panel ──
ax4 = axes[1, 1]
ax4.axis("off")
summary_text = (
    f"{'RLHF TRAINING SUMMARY':^40}\n"
    f"{'─' * 40}\n\n"
    f"  Model             : GPT-2 (124M)\n"
    f"  Reward Model      : DistilBERT-SST2\n"
    f"  Algorithm         : REINFORCE + KL\n"
    f"  Training Prompts  : {len(TRAIN_PROMPTS)}\n"
    f"  Epochs            : {NUM_EPOCHS}\n"
    f"  Learning Rate     : {LEARNING_RATE}\n"
    f"  KL Coefficient    : {KL_COEFF}\n\n"
    f"{'─' * 40}\n"
    f"  Avg Reward BEFORE : {df_before['reward'].mean():+.4f}\n"
    f"  Avg Reward AFTER  : {df_after['reward'].mean():+.4f}\n"
    f"  Improvement (Δ)   : {df_after['reward'].mean() - df_before['reward'].mean():+.4f}\n"
    f"{'─' * 40}\n\n"
    f"  Prompts improved  : {improved} / {len(comparison)}\n"
    f"  Prompts worsened  : {worsened} / {len(comparison)}\n"
    f"  Best Δ            : {comparison['Delta'].max():+.4f}\n"
    f"  Worst Δ           : {comparison['Delta'].min():+.4f}\n"
)
ax4.text(0.05, 0.95, summary_text, transform=ax4.transAxes,
         fontsize=11, verticalalignment="top", fontfamily="monospace",
         bbox=dict(boxstyle="round,pad=0.8", facecolor="#F7F7F7", edgecolor="#CCCCCC"))

plt.tight_layout()
plt.savefig("rlhf_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Plot saved to rlhf_results.png")

---
## Cell 10 — Try Your Own Prompts

In [ ]:
print("\n" + "=" * 70)
print("  🎯  CUSTOM PROMPTS — Before vs After RLHF")
print("=" * 70)

# ✏️ Edit these prompts to try your own!
custom_prompts = [
    "The customer service was",
    "I strongly believe that education",
    "This software update",
]

for prompt in custom_prompts:
    gen_before = generate_text(ref_model, prompt)
    score_before = get_reward(gen_before)

    gen_after = generate_text(policy_model, prompt)
    score_after = get_reward(gen_after)

    print(f"\n  Prompt: \"{prompt}\"")
    print(f"  ┌─ BEFORE RLHF ─────────────────────────────────────")
    print(f"  │  {textwrap.fill(gen_before, width=60, subsequent_indent='  │  ')}")
    print(f"  │  Reward: {score_before:+.4f}")
    print(f"  ├─ AFTER RLHF ──────────────────────────────────────")
    print(f"  │  {textwrap.fill(gen_after, width=60, subsequent_indent='  │  ')}")
    print(f"  │  Reward: {score_after:+.4f}")
    print(f"  └───────────────────────────────────────────────────")

print("\n✅ Demo Complete! Edit 'custom_prompts' above to try your own.\n")

---
## 📝 Summary

### What this notebook demonstrated:

| Stage | What happens |
|-------|-------------|
| **Before RLHF** | GPT-2 generates freely — output sentiment is uncontrolled |
| **Reward signal** | DistilBERT-SST2 scores each generation by sentiment |
| **RLHF training** | REINFORCE + KL penalty optimizes GPT-2 to maximize reward while staying close to the original model |
| **After RLHF** | GPT-2 is steered toward generating more positive text |

### Key hyperparameters to experiment with:

| Parameter | Current | Effect of increasing |
|-----------|---------|---------------------|
| `NUM_EPOCHS` | 3 | Stronger alignment, but risk of reward hacking |
| `KL_COEFF` | 0.15 | Keeps outputs closer to original GPT-2 (less drift) |
| `LEARNING_RATE` | 1.41e-5 | Faster learning but less stable |
| `TEMPERATURE` | 0.8 | More diverse/creative outputs |

### Swap the reward model:
Replace `REWARD_NAME` with any HuggingFace classifier to steer toward different objectives:
- Toxicity reduction
- Formality
- Topic relevance
- Factuality